<a href="https://colab.research.google.com/github/MadhaviSG/image-music/blob/main/Image_To_Music.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image → Music Generation
CLIP image encoder → Projection Layer → MusicGen decoder

**Run order:** Cells 1–14 setup → Cell 15 pre-training baselines → Cell 16 train → Cells 17–22 evaluate → Cell 23 demo

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install torch torchaudio transformers gradio Pillow scipy pillow-avif-plugin wandb --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 15.0 MB/s eta 0:00:00


In [4]:
import os, csv, gc, scipy.io.wavfile, scipy.signal
import numpy as np
import torch
import torch.nn as nn
import torchaudio
import matplotlib.pyplot as plt
import wandb
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoProcessor, MusicgenForConditionalGeneration,
    CLIPModel, CLIPProcessor,
)
from transformers.modeling_outputs import BaseModelOutput
import pillow_avif  # registers .avif support with PIL

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

Device: cuda


In [6]:
wandb.login()  # paste your API key when prompted

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mgulavan (mgulavan-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Load Models

In [7]:
clip_model     = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

musicgen           = MusicgenForConditionalGeneration.from_pretrained('facebook/musicgen-small').to(device)
musicgen_processor = AutoProcessor.from_pretrained('facebook/musicgen-small')

for cfg in [musicgen.config, musicgen.decoder.config, musicgen.generation_config]:
    cfg.decoder_start_token_id = 2048
    cfg.pad_token_id           = 2048

CLIP_DIM     = clip_model.config.projection_dim      # 512
TEXT_ENC_DIM = musicgen.config.text_encoder.d_model  # 768
print(f'CLIP_DIM={CLIP_DIM}  TEXT_ENC_DIM={TEXT_ENC_DIM}')
print(f'Audio SR: {musicgen.config.audio_encoder.sampling_rate}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

CLIP_DIM=512  TEXT_ENC_DIM=768
Audio SR: 32000


In [8]:
for p in clip_model.parameters():              p.requires_grad = False
for p in musicgen.audio_encoder.parameters():  p.requires_grad = False
for p in musicgen.text_encoder.parameters():   p.requires_grad = False
for p in musicgen.enc_to_dec_proj.parameters(): p.requires_grad = True
for p in musicgen.decoder.parameters():         p.requires_grad = True

trainable = sum(p.numel() for p in musicgen.parameters() if p.requires_grad)
total     = sum(p.numel() for p in musicgen.parameters())
print(f'MusicGen trainable: {trainable:,} / {total:,}')

MusicGen trainable: 420,371,456 / 586,884,674


## Model Architecture

In [9]:
# INSTRUMENT_NAMES = [
#     # Indian (0-13)
#     'sitar','tabla','sarod','veena','bansuri','shehnai',
#     'mridangam','dhol','dholak','tasha','sarangi','tanpura','harmonium','santoor',
#     # Middle East (14-24)
#     'oud','qanun','ney','darbuka','riq','rabab','buzuq','saz','bendir','zurna','kamanche',
#     # East Asia (25-36)
#     'guzheng','erhu','pipa','dizi','guqin','taiko','shamisen','koto','shakuhachi','gayageum','suona','sheng',
# ]
# NUM_INSTRUMENTS = len(INSTRUMENT_NAMES)  # 37

# CULTURE_INSTRUMENTS = {
#     'indian':      list(range(0,  14)),
#     'middle_east': list(range(14, 25)),
#     'east_asia':   list(range(25, 37)),
# }

# SEQ_LEN = 8

# class ImageProjection(nn.Module):
#     def __init__(self, clip_dim, text_enc_dim, seq_len):
#         super().__init__()
#         self.seq_len = seq_len
#         self.net = nn.Sequential(
#             nn.Linear(clip_dim, text_enc_dim * 2),
#             nn.GELU(),
#             nn.LayerNorm(text_enc_dim * 2),
#             nn.Linear(text_enc_dim * 2, text_enc_dim * seq_len),
#         )
#     def forward(self, x):
#         return self.net(x).view(x.shape[0], self.seq_len, -1)

# class InstrumentClassifier(nn.Module):
#     def __init__(self, clip_dim, num_instruments):
#         super().__init__()
#         self.fc = nn.Linear(clip_dim, num_instruments)
#     def forward(self, x):
#         return self.fc(x)

# projection = ImageProjection(CLIP_DIM, TEXT_ENC_DIM, SEQ_LEN).to(device)
# inst_clf   = InstrumentClassifier(CLIP_DIM, NUM_INSTRUMENTS).to(device)
# print(f'Projection: {sum(p.numel() for p in projection.parameters()):,} params')
# print(f'Classifier: {sum(p.numel() for p in inst_clf.parameters()):,} params')

Projection: 10,234,368 params
Classifier: 18,981 params


In [19]:
INSTRUMENT_NAMES = [
    'sitar','tabla','sarod','veena','bansuri','shehnai',
    'mridangam','dhol','dholak','tasha','sarangi','tanpura','harmonium','santoor',
    'oud','qanun','ney','darbuka','riq','rabab','buzuq','saz','bendir','zurna','kamanche',
    'guzheng','erhu','pipa','dizi','guqin','taiko','shamisen','koto','shakuhachi','gayageum','suona','sheng',
]
NUM_INSTRUMENTS = len(INSTRUMENT_NAMES)

CULTURE_INSTRUMENTS = {
    'indian':      list(range(0,  14)),
    'middle_east': list(range(14, 25)),
    'east_asia':   list(range(25, 37)),
}

SEQ_LEN = 8

class ImageProjection(nn.Module):
    def __init__(self, clip_dim, text_enc_dim, seq_len, dropout=0.1):
        super().__init__()
        self.seq_len = seq_len
        self.net = nn.Sequential(
            nn.Linear(clip_dim, text_enc_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(text_enc_dim * 2),
            nn.Linear(text_enc_dim * 2, text_enc_dim * seq_len),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x).view(x.shape[0], self.seq_len, -1)

class InstrumentClassifier(nn.Module):
    def __init__(self, clip_dim, num_instruments):
        super().__init__()
        self.fc = nn.Linear(clip_dim, num_instruments)
    def forward(self, x):
        return self.fc(x)

projection = ImageProjection(CLIP_DIM, TEXT_ENC_DIM, SEQ_LEN).to(device)
inst_clf   = InstrumentClassifier(CLIP_DIM, NUM_INSTRUMENTS).to(device)
print(f'Projection: {sum(p.numel() for p in projection.parameters()):,} params')
print(f'Classifier: {sum(p.numel() for p in inst_clf.parameters()):,} params')

Projection: 10,234,368 params
Classifier: 18,981 params


## Dataset

In [20]:
class ImageAudioDataset(Dataset):
    def __init__(self, csv_path, target_sr=32000, duration=5):
        with open(csv_path, newline='') as f:
            self.data = list(csv.DictReader(f))
        self.target_sr   = target_sr
        self.num_samples = target_sr * duration

    def __len__(self): return len(self.data)

    def _load_audio(self, path):
        waveform, sr = torchaudio.load(path)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sr != self.target_sr:
            waveform = torchaudio.transforms.Resample(sr, self.target_sr)(waveform)
        if waveform.shape[1] >= self.num_samples:
            waveform = waveform[:, :self.num_samples]
        else:
            waveform = nn.functional.pad(waveform, (0, self.num_samples - waveform.shape[1]))
        return waveform

    def __getitem__(self, idx):
        row     = self.data[idx]
        image   = Image.open(row['image_path']).convert('RGB')
        audio   = self._load_audio(row['music_path'])
        culture = row['culture'].strip().lower()
        inst_label = torch.zeros(NUM_INSTRUMENTS)
        for i in CULTURE_INSTRUMENTS.get(culture, []):
            inst_label[i] = 1.0
        return {'image': image, 'audio': audio, 'inst_label': inst_label, 'culture': culture}

def collate_fn(batch):
    return {
        'images':      [x['image'] for x in batch],
        'audio':       torch.stack([x['audio'] for x in batch]),
        'inst_labels': torch.stack([x['inst_label'] for x in batch]),
        'cultures':    [x['culture'] for x in batch],
    }


In [21]:
CSV_PATH = '/content/drive/MyDrive/18789 Project/Dataset/dataset.csv'

dataset    = ImageAudioDataset(CSV_PATH, target_sr=32000, duration=5)
val_size   = max(3, int(0.1 * len(dataset)))
train_size = len(dataset) - val_size
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

cultures_in_train = [dataset.data[i]['culture'].strip().lower() for i in train_ds.indices]
culture_counts    = {c: cultures_in_train.count(c) for c in set(cultures_in_train)}
print('Training pairs per culture:', culture_counts)
weights = [1.0 / culture_counts[c] for c in cultures_in_train]
sampler = torch.utils.data.WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=2, sampler=sampler, collate_fn=collate_fn,
                          num_workers=2, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=2, shuffle=False,   collate_fn=collate_fn,
                          num_workers=2, pin_memory=True, persistent_workers=True)
print(f'Train: {train_size}  Val: {val_size}')

# confirm GPU
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE — check runtime type!"}')

Training pairs per culture: {'east_asia': 92, 'indian': 87, 'middle_east': 91}
Train: 270  Val: 30
GPU: Tesla T4


In [22]:
CSV_PATH = '/content/drive/MyDrive/18789 Project/Dataset/dataset.csv'

dataset    = ImageAudioDataset(CSV_PATH, target_sr=32000, duration=5)
val_size   = max(3, int(0.1 * len(dataset)))
train_size = len(dataset) - val_size
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

cultures_in_train = [dataset.data[i]['culture'].strip().lower() for i in train_ds.indices]
culture_counts    = {c: cultures_in_train.count(c) for c in set(cultures_in_train)}
print('Training pairs per culture:', culture_counts)
weights = [1.0 / culture_counts[c] for c in cultures_in_train]
sampler = torch.utils.data.WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=2, sampler=sampler,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=2, shuffle=False,    collate_fn=collate_fn)
print(f'Train: {train_size}  Val: {val_size}')

Training pairs per culture: {'east_asia': 89, 'indian': 89, 'middle_east': 92}
Train: 270  Val: 30


## Helpers

In [23]:
def encode_audio_to_labels(audio_batch):
    B = audio_batch.shape[0]
    with torch.no_grad():
        encoded = musicgen.audio_encoder.encode(
            audio_batch, bandwidth=musicgen.audio_encoder.config.target_bandwidths[0]
        )
    audio_codes = encoded.audio_codes
    if audio_codes.dim() != 4:
        raise ValueError(f'Unexpected audio_codes shape: {audio_codes.shape}')
    if audio_codes.shape[0] == B:
        B_, chunks, K, frames = audio_codes.shape
        labels = audio_codes.permute(0,1,3,2).contiguous().view(B_, chunks*frames, K)
    elif audio_codes.shape[1] == B:
        chunks, B_, K, frames = audio_codes.shape
        labels = audio_codes.permute(1,0,3,2).contiguous().view(B_, chunks*frames, K)
    else:
        raise ValueError(f'Cannot identify batch dim. Shape: {audio_codes.shape}, B={B}')
    return labels.long()

@torch.no_grad()
def get_clip_features(images):
    inputs     = clip_processor(images=images, return_tensors='pt').to(device)
    vision_out = clip_model.vision_model(**inputs)
    feats      = clip_model.visual_projection(vision_out.pooler_output)
    return feats / (feats.norm(dim=-1, keepdim=True) + 1e-8)

def image_to_encoder_outputs(image_path):
    image      = Image.open(image_path).convert('RGB')
    clip_feats = get_clip_features([image])
    proj       = projection(clip_feats)
    enc_out    = BaseModelOutput(last_hidden_state=proj)
    attn_mask  = torch.ones(proj.shape[:2], dtype=torch.long, device=device)
    return clip_feats, enc_out, attn_mask

def save_wav(audio_np, sr, path):
    scipy.io.wavfile.write(path, sr, audio_np)

print('Helpers defined.')

Helpers defined.


## Pre-Training Baselines
**Run this BEFORE the training cell.** Saves Baseline A (untrained image model) and Baseline B (text prompt).

In [16]:
EVAL_DURATION = 10
EVAL_DIR      = '/content/drive/MyDrive/18789 Project/eval_outputs'
os.makedirs(EVAL_DIR, exist_ok=True)

CULTURE_TEXT_PROMPTS = {
    'indian':      'Indian classical music with sitar and tabla',
    'middle_east': 'traditional Arabic music with oud and qanun',
    'east_asia':   'traditional Chinese music with guzheng and erhu',
}

# Pick one held-out sample per culture from val set
eval_samples = {}
for i in val_ds.indices:
    row     = dataset.data[i]
    culture = row['culture'].strip().lower()
    if culture not in eval_samples:
        eval_samples[culture] = {'image_path': row['image_path'], 'audio_path': row['music_path']}
    if len(eval_samples) == 3:
        break

print('Eval samples:')
for c, s in eval_samples.items():
    print(f'  {c}: {s["image_path"]}')

def save_wav(audio_np, sr, path):
    scipy.io.wavfile.write(path, sr, audio_np)

def generate_from_image_raw(image_path, duration=EVAL_DURATION):
    """Generate audio from image using current model weights. guidance_scale=1.0 (CFG disabled — incompatible with custom encoder_outputs)."""
    musicgen.eval(); projection.eval()
    with torch.no_grad():
        _, enc_out, attn_mask = image_to_encoder_outputs(image_path)
        audio = musicgen.generate(
            encoder_outputs=enc_out,
            attention_mask=attn_mask,
            do_sample=True,
            guidance_scale=1.0,
            max_new_tokens=int(duration * musicgen.config.audio_encoder.frame_rate),
        )
    sr = musicgen.config.audio_encoder.sampling_rate
    return audio[0, 0].cpu().numpy(), sr

def generate_from_text_raw(culture, duration=EVAL_DURATION):
    """Generate audio from text prompt. guidance_scale=3.0 (CFG works fine for text path)."""
    musicgen.eval()
    prompt = CULTURE_TEXT_PROMPTS[culture]
    inputs = musicgen_processor(text=[prompt], padding=True, return_tensors='pt').to(device)
    with torch.no_grad():
        audio = musicgen.generate(
            **inputs, do_sample=True, guidance_scale=3.0,
            max_new_tokens=int(duration * musicgen.config.audio_encoder.frame_rate),
        )
    sr = musicgen.config.audio_encoder.sampling_rate
    return audio[0, 0].cpu().numpy(), sr

print('\nGenerating pre-training baselines...')
for culture, sample in eval_samples.items():
    a_np, a_sr = generate_from_image_raw(sample['image_path'])
    save_wav(a_np, a_sr, f'{EVAL_DIR}/{culture}_baseline_a_pretrain.wav')

    b_np, b_sr = generate_from_text_raw(culture)
    save_wav(b_np, b_sr, f'{EVAL_DIR}/{culture}_baseline_b_text.wav')
    print(f'  {culture}: done')

print('Baselines saved to', EVAL_DIR)

Eval samples:
  indian: /content/drive/MyDrive/18789 Project/Dataset/India/imgs/IND_106.jpg
  middle_east: /content/drive/MyDrive/18789 Project/Dataset/Middle_East/imgs/44.jpg
  east_asia: /content/drive/MyDrive/18789 Project/Dataset/East_Asia/imgs/11.jpg

Generating pre-training baselines...
  indian: done
  middle_east: done
  east_asia: done
Baselines saved to /content/drive/MyDrive/18789 Project/eval_outputs


## Training

In [18]:
# # 1
# from tqdm.notebook import tqdm as tqdm_nb

# EPOCHS           = 50
# LR               = 1e-4
# INST_LOSS_WEIGHT = 0.1
# SAVE_DIR         = '/content/drive/MyDrive/18789 Project/image_to_music_checkpoint'
# os.makedirs(SAVE_DIR, exist_ok=True)

# run = wandb.init(
#     project='image-to-music',
#     config={
#         'epochs': EPOCHS, 'lr': LR, 'batch_size': 2,
#         'seq_len': SEQ_LEN, 'inst_loss_weight': INST_LOSS_WEIGHT,
#         'clip_dim': CLIP_DIM, 'text_enc_dim': TEXT_ENC_DIM,
#         'train_size': train_size, 'val_size': val_size,
#     }
# )

# optimizer   = torch.optim.AdamW(
#     list(projection.parameters()) +
#     list(inst_clf.parameters()) +
#     list(filter(lambda p: p.requires_grad, musicgen.parameters())),
#     lr=LR, weight_decay=1e-4,
# )
# clf_loss_fn = nn.BCEWithLogitsLoss()
# scaler      = torch.amp.GradScaler('cuda')
# torch.backends.cudnn.benchmark = True

# train_losses, val_losses = [], []
# best_val_loss = float('inf')

# def save_checkpoint(epoch, tag):
#     musicgen.save_pretrained(SAVE_DIR)
#     musicgen_processor.save_pretrained(SAVE_DIR)
#     torch.save({
#         'projection':       projection.state_dict(),
#         'inst_clf':         inst_clf.state_dict(),
#         'clip_dim':         CLIP_DIM,
#         'text_enc_dim':     TEXT_ENC_DIM,
#         'seq_len':          SEQ_LEN,
#         'instrument_names': INSTRUMENT_NAMES,
#         'train_losses':     train_losses,
#         'val_losses':       val_losses,
#         'epoch':            epoch,
#     }, os.path.join(SAVE_DIR, 'heads.pt'))
#     artifact = wandb.Artifact(
#         name=f'image-to-music-{tag}', type='model',
#         metadata={'epoch': epoch, 'tag': tag,
#                   'train_loss': train_losses[-1] if train_losses else None,
#                   'val_loss':   val_losses[-1]   if val_losses   else None}
#     )
#     artifact.add_dir(SAVE_DIR)
#     wandb.log_artifact(artifact)
#     print(f'  [{tag}] checkpoint saved to Drive + WandB (epoch {epoch})')

# # ── outer bar: one tick per epoch ──────────────────────────────────────────
# epoch_bar = tqdm_nb(range(EPOCHS), desc='Training', unit='epoch')

# for epoch in epoch_bar:
#     musicgen.train(); projection.train(); inst_clf.train()
#     epoch_loss  = 0.0
#     music_loss_sum = 0.0
#     inst_loss_sum  = 0.0

#     # ── inner bar: one tick per batch ──────────────────────────────────────
#     batch_bar = tqdm_nb(train_loader, desc=f'Ep {epoch+1:02d} train',
#                         leave=False, unit='batch')
#     for step, batch in enumerate(batch_bar):
#         audio       = batch['audio'].to(device, non_blocking=True)
#         inst_labels = batch['inst_labels'].to(device, non_blocking=True)
#         with torch.amp.autocast('cuda'):
#             clip_feats = get_clip_features(batch['images'])
#             proj       = projection(clip_feats)
#             enc_out    = BaseModelOutput(last_hidden_state=proj)
#             attn_mask  = torch.ones(proj.shape[:2], dtype=torch.long, device=device)
#             labels     = encode_audio_to_labels(audio).to(device)
#             ml         = musicgen(encoder_outputs=enc_out, attention_mask=attn_mask, labels=labels).loss
#             il         = clf_loss_fn(inst_clf(clip_feats.detach()), inst_labels)
#             loss       = ml + INST_LOSS_WEIGHT * il
#         optimizer.zero_grad()
#         scaler.scale(loss).backward()
#         scaler.unscale_(optimizer)
#         torch.nn.utils.clip_grad_norm_(
#             list(projection.parameters()) + list(musicgen.parameters()), 1.0
#         )
#         scaler.step(optimizer)
#         scaler.update()

#         epoch_loss     += loss.item()
#         music_loss_sum += ml.item()
#         inst_loss_sum  += il.item()

#         # live per-batch stats in the inner bar
#         batch_bar.set_postfix(
#             loss=f'{loss.item():.4f}',
#             music=f'{ml.item():.4f}',
#             inst=f'{il.item():.4f}',
#         )

#     avg_train = epoch_loss     / len(train_loader)
#     avg_music = music_loss_sum / len(train_loader)
#     avg_inst  = inst_loss_sum  / len(train_loader)
#     train_losses.append(avg_train)

#     # ── validation ─────────────────────────────────────────────────────────
#     musicgen.eval(); projection.eval(); inst_clf.eval()
#     val_loss = 0.0
#     val_bar  = tqdm_nb(val_loader, desc=f'Ep {epoch+1:02d} val  ',
#                        leave=False, unit='batch')
#     with torch.no_grad(), torch.amp.autocast('cuda'):
#         for batch in val_bar:
#             audio       = batch['audio'].to(device, non_blocking=True)
#             inst_labels = batch['inst_labels'].to(device, non_blocking=True)
#             clip_feats  = get_clip_features(batch['images'])
#             proj        = projection(clip_feats)
#             enc_out     = BaseModelOutput(last_hidden_state=proj)
#             attn_mask   = torch.ones(proj.shape[:2], dtype=torch.long, device=device)
#             labels      = encode_audio_to_labels(audio).to(device)
#             out         = musicgen(encoder_outputs=enc_out, attention_mask=attn_mask, labels=labels)
#             vl          = out.loss + INST_LOSS_WEIGHT * clf_loss_fn(inst_clf(clip_feats), inst_labels)
#             val_loss   += vl.item()
#             val_bar.set_postfix(val_loss=f'{vl.item():.4f}')

#     avg_val = val_loss / len(val_loader)
#     val_losses.append(avg_val)
#     is_best = avg_val < best_val_loss
#     if is_best:
#         best_val_loss = avg_val

#     # ── update outer bar with epoch summary ────────────────────────────────
#     epoch_bar.set_postfix(
#         train=f'{avg_train:.4f}',
#         music=f'{avg_music:.4f}',
#         inst=f'{avg_inst:.4f}',
#         val=f'{avg_val:.4f}',
#         best=f'{best_val_loss:.4f}',
#         new_best='★' if is_best else '',
#     )

#     log_dict = {
#         'train_loss': avg_train, 'train_music_loss': avg_music,
#         'train_inst_loss': avg_inst, 'val_loss': avg_val,
#         'best_val_loss': best_val_loss, 'epoch': epoch + 1,
#     }

#     if (epoch + 1) % 10 == 0:
#         print(f'\n── Epoch {epoch+1} ── train={avg_train:.4f}  music={avg_music:.4f}  inst={avg_inst:.4f}  val={avg_val:.4f}  best={best_val_loss:.4f}')
#         for culture, sample in eval_samples.items():
#             a_np, a_sr = generate_from_image_raw(sample['image_path'], duration=5)
#             log_dict[f'audio_{culture}'] = wandb.Audio(a_np, sample_rate=a_sr, caption=f'ep{epoch+1}')
#         save_checkpoint(epoch + 1, tag=f'ep{epoch+1}')
#         musicgen.train(); projection.train(); inst_clf.train()

#     if is_best:
#         save_checkpoint(epoch + 1, tag='best')
#         musicgen.train(); projection.train(); inst_clf.train()

#     wandb.log(log_dict)

# save_checkpoint(EPOCHS, tag='last')
# wandb.finish()
# print('\nTraining done.')

Training:   0%|          | 0/50 [00:00<?, ?epoch/s]

Ep 01 train:   0%|          | 0/135 [00:00<?, ?batch/s]

Ep 01 val  :   0%|          | 0/15 [00:00<?, ?batch/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

wandb: Adding directory to artifact (/content/drive/MyDrive/18789 Project/image_to_music_checkpoint)... Done. 114.6s


...

Ep 26 val  :   0%|          | 0/15 [00:00<?, ?batch/s]

Ep 27 train:   0%|          | 0/135 [00:00<?, ?batch/s]

Ep 27 val  :   0%|          | 0/15 [00:00<?, ?batch/s]

Ep 28 train:   0%|          | 0/135 [00:00<?, ?batch/s]

KeyboardInterrupt: 

In [ ]:
from tqdm.notebook import tqdm as tqdm_nb

EPOCHS           = 50
LR               = 1e-4
INST_LOSS_WEIGHT = 0.1
SAVE_DIR         = '/content/drive/MyDrive/18789 Project/image_to_music_checkpoint'
os.makedirs(SAVE_DIR, exist_ok=True)

run = wandb.init(
    project='image-to-music',
    config={
        'epochs': EPOCHS, 'lr': LR, 'batch_size': 2,
        'seq_len': SEQ_LEN, 'inst_loss_weight': INST_LOSS_WEIGHT,
        'clip_dim': CLIP_DIM, 'text_enc_dim': TEXT_ENC_DIM,
        'train_size': train_size, 'val_size': val_size,
        'dropout': 0.1, 'scheduler': 'cosine',
    }
)

optimizer = torch.optim.AdamW(
    list(projection.parameters()) +
    list(inst_clf.parameters()) +
    list(filter(lambda p: p.requires_grad, musicgen.parameters())),
    lr=LR, weight_decay=1e-4,
)
scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
clf_loss_fn = nn.BCEWithLogitsLoss()
scaler      = torch.amp.GradScaler('cuda')
torch.backends.cudnn.benchmark = True

train_losses, val_losses = [], []
best_val_loss = float('inf')

def save_checkpoint(epoch, tag):
    musicgen.save_pretrained(SAVE_DIR)
    musicgen_processor.save_pretrained(SAVE_DIR)
    torch.save({
        'projection':       projection.state_dict(),
        'inst_clf':         inst_clf.state_dict(),
        'clip_dim':         CLIP_DIM,
        'text_enc_dim':     TEXT_ENC_DIM,
        'seq_len':          SEQ_LEN,
        'instrument_names': INSTRUMENT_NAMES,
        'train_losses':     train_losses,
        'val_losses':       val_losses,
        'epoch':            epoch,
    }, os.path.join(SAVE_DIR, 'heads.pt'))
    artifact = wandb.Artifact(
        name=f'image-to-music-{tag}', type='model',
        metadata={'epoch': epoch, 'tag': tag,
                  'train_loss': train_losses[-1] if train_losses else None,
                  'val_loss':   val_losses[-1]   if val_losses   else None}
    )
    artifact.add_dir(SAVE_DIR)
    wandb.log_artifact(artifact)
    print(f'  [{tag}] checkpoint saved to Drive + WandB (epoch {epoch})')

epoch_bar = tqdm_nb(range(EPOCHS), desc='Training', unit='epoch')

for epoch in epoch_bar:
    musicgen.train(); projection.train(); inst_clf.train()
    epoch_loss = music_loss_sum = inst_loss_sum = 0.0

    batch_bar = tqdm_nb(train_loader, desc=f'Ep {epoch+1:02d} train',
                        leave=False, unit='batch')
    for batch in batch_bar:
        audio       = batch['audio'].to(device, non_blocking=True)
        inst_labels = batch['inst_labels'].to(device, non_blocking=True)
        with torch.amp.autocast('cuda'):
            clip_feats = get_clip_features(batch['images'])
            proj       = projection(clip_feats)
            enc_out    = BaseModelOutput(last_hidden_state=proj)
            attn_mask  = torch.ones(proj.shape[:2], dtype=torch.long, device=device)
            labels     = encode_audio_to_labels(audio).to(device)
            ml         = musicgen(encoder_outputs=enc_out, attention_mask=attn_mask, labels=labels).loss
            il         = clf_loss_fn(inst_clf(clip_feats.detach()), inst_labels)
            loss       = ml + INST_LOSS_WEIGHT * il
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            list(projection.parameters()) + list(musicgen.parameters()), 1.0
        )
        scaler.step(optimizer)
        scaler.update()
        epoch_loss     += loss.item()
        music_loss_sum += ml.item()
        inst_loss_sum  += il.item()
        batch_bar.set_postfix(loss=f'{loss.item():.4f}', music=f'{ml.item():.4f}', inst=f'{il.item():.4f}')

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    avg_train = epoch_loss     / len(train_loader)
    avg_music = music_loss_sum / len(train_loader)
    avg_inst  = inst_loss_sum  / len(train_loader)
    train_losses.append(avg_train)

    musicgen.eval(); projection.eval(); inst_clf.eval()
    val_loss = 0.0
    val_bar  = tqdm_nb(val_loader, desc=f'Ep {epoch+1:02d} val  ',
                       leave=False, unit='batch')
    with torch.no_grad(), torch.amp.autocast('cuda'):
        for batch in val_bar:
            audio       = batch['audio'].to(device, non_blocking=True)
            inst_labels = batch['inst_labels'].to(device, non_blocking=True)
            clip_feats  = get_clip_features(batch['images'])
            proj        = projection(clip_feats)
            enc_out     = BaseModelOutput(last_hidden_state=proj)
            attn_mask   = torch.ones(proj.shape[:2], dtype=torch.long, device=device)
            labels      = encode_audio_to_labels(audio).to(device)
            out         = musicgen(encoder_outputs=enc_out, attention_mask=attn_mask, labels=labels)
            vl          = out.loss + INST_LOSS_WEIGHT * clf_loss_fn(inst_clf(clip_feats), inst_labels)
            val_loss   += vl.item()
            val_bar.set_postfix(val_loss=f'{vl.item():.4f}')

    avg_val = val_loss / len(val_loader)
    val_losses.append(avg_val)
    is_best = avg_val < best_val_loss
    if is_best:
        best_val_loss = avg_val

    epoch_bar.set_postfix(
        train=f'{avg_train:.4f}', music=f'{avg_music:.4f}',
        inst=f'{avg_inst:.4f}',   val=f'{avg_val:.4f}',
        best=f'{best_val_loss:.4f}', lr=f'{current_lr:.2e}',
        new_best='★' if is_best else '',
    )

    log_dict = {
        'train_loss': avg_train, 'train_music_loss': avg_music,
        'train_inst_loss': avg_inst, 'val_loss': avg_val,
        'best_val_loss': best_val_loss, 'lr': current_lr, 'epoch': epoch + 1,
    }

    if (epoch + 1) % 10 == 0:
        print(f'\n── Epoch {epoch+1} ── train={avg_train:.4f}  music={avg_music:.4f}  inst={avg_inst:.4f}  val={avg_val:.4f}  best={best_val_loss:.4f}  lr={current_lr:.2e}')
        for culture, sample in eval_samples.items():
            a_np, a_sr = generate_from_image_raw(sample['image_path'], duration=5)
            log_dict[f'audio_{culture}'] = wandb.Audio(a_np, sample_rate=a_sr, caption=f'ep{epoch+1}')
        save_checkpoint(epoch + 1, tag=f'ep{epoch+1}')
        musicgen.train(); projection.train(); inst_clf.train()

    if is_best:
        save_checkpoint(epoch + 1, tag='best')
        musicgen.train(); projection.train(); inst_clf.train()

    wandb.log(log_dict)

save_checkpoint(EPOCHS, tag='last')
wandb.finish()
print('\nTraining done.')

best_val_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▂▂▂▂▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇██
train_inst_loss,██▇▇▇▆▆▆▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁
train_loss,█▇▇▇▇▆▆▆▅▅▅▅▄▄▃▃▃▃▃▃▂▂▂▁▁▁▁
train_music_loss,█▇▇▇▇▆▆▆▅▅▅▅▄▄▃▃▃▃▃▃▂▂▂▁▁▁▁
val_loss,▂▁▁▂▂▃▄▂▂▃▂▃▃▃▃▅▃▄▅▅▅▅▅▇▆██
best_val_loss,6.73007
epoch,27
train_inst_loss,0.4353
train_loss,4.76493
train_music_loss,4.7214


Training:   0%|          | 0/50 [00:00<?, ?epoch/s]

Ep 01 train:   0%|          | 0/135 [00:00<?, ?batch/s]

Ep 01 val  :   0%|          | 0/15 [00:00<?, ?batch/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

wandb: Adding directory to artifact (/content/drive/MyDrive/18789 Project/image_to_music_checkpoint)... Done. 110.4s


  [ep10] checkpoint saved to Drive + WandB (epoch 10)


Ep 11 train:   0%|          | 0/135 [00:00<?, ?batch/s]

Ep 11 val  :   0%|          | 0/15 [00:00<?, ?batch/s]

Ep 12 train:   0%|          | 0/135 [00:00<?, ?batch/s]

## Training

In [ ]:
# EPOCHS           = 50
# LR               = 1e-4
# INST_LOSS_WEIGHT = 0.1
# SAVE_DIR         = '/content/drive/MyDrive/18789 Project/image_to_music_checkpoint'

# run = wandb.init(
#     project='image-to-music',
#     config={
#         'epochs': EPOCHS, 'lr': LR, 'batch_size': 2,
#         'seq_len': SEQ_LEN, 'inst_loss_weight': INST_LOSS_WEIGHT,
#         'clip_dim': CLIP_DIM, 'text_enc_dim': TEXT_ENC_DIM,
#         'train_size': train_size, 'val_size': val_size,
#     }
# )

# optimizer   = torch.optim.AdamW(
#     list(projection.parameters()) +
#     list(inst_clf.parameters()) +
#     list(filter(lambda p: p.requires_grad, musicgen.parameters())),
#     lr=LR, weight_decay=1e-4,
# )
# clf_loss_fn = nn.BCEWithLogitsLoss()
# train_losses, val_losses = [], []

# for epoch in range(EPOCHS):
#     # ── train ──
#     musicgen.train(); projection.train(); inst_clf.train()
#     epoch_loss = 0.0
#     for batch in train_loader:
#         audio       = batch['audio'].to(device)
#         inst_labels = batch['inst_labels'].to(device)

#         clip_feats  = get_clip_features(batch['images'])
#         proj        = projection(clip_feats)
#         enc_out     = BaseModelOutput(last_hidden_state=proj)
#         attn_mask   = torch.ones(proj.shape[:2], dtype=torch.long, device=device)
#         labels      = encode_audio_to_labels(audio).to(device)

#         music_loss  = musicgen(encoder_outputs=enc_out, attention_mask=attn_mask, labels=labels).loss
#         inst_loss   = clf_loss_fn(inst_clf(clip_feats.detach()), inst_labels)
#         loss        = music_loss + INST_LOSS_WEIGHT * inst_loss

#         optimizer.zero_grad()
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(
#             list(projection.parameters()) + list(musicgen.parameters()), 1.0
#         )
#         optimizer.step()
#         epoch_loss += loss.item()

#     avg_train = epoch_loss / len(train_loader)
#     train_losses.append(avg_train)

#     # ── val ──
#     musicgen.eval(); projection.eval(); inst_clf.eval()
#     val_loss = 0.0
#     with torch.no_grad():
#         for batch in val_loader:
#             audio       = batch['audio'].to(device)
#             inst_labels = batch['inst_labels'].to(device)
#             clip_feats  = get_clip_features(batch['images'])
#             proj        = projection(clip_feats)
#             enc_out     = BaseModelOutput(last_hidden_state=proj)
#             attn_mask   = torch.ones(proj.shape[:2], dtype=torch.long, device=device)
#             labels      = encode_audio_to_labels(audio).to(device)
#             out         = musicgen(encoder_outputs=enc_out, attention_mask=attn_mask, labels=labels)
#             val_loss   += (out.loss + INST_LOSS_WEIGHT * clf_loss_fn(inst_clf(clip_feats), inst_labels)).item()

#     avg_val = val_loss / len(val_loader)
#     val_losses.append(avg_val)

#     wandb.log({'train_loss': avg_train, 'val_loss': avg_val, 'epoch': epoch + 1})

#     # log generated audio sample every 10 epochs
#     if (epoch + 1) % 10 == 0:
#         sample_culture = list(eval_samples.keys())[0]
#         a_np, a_sr = generate_from_image_raw(eval_samples[sample_culture]['image_path'], duration=5)
#         wandb.log({f'audio_{sample_culture}_ep{epoch+1}': wandb.Audio(a_np, sample_rate=a_sr)})
#         musicgen.train(); projection.train(); inst_clf.train()

#     if (epoch + 1) % 5 == 0:
#         print(f'Epoch {epoch+1:3d}/{EPOCHS}  train={avg_train:.4f}  val={avg_val:.4f}')

# wandb.finish()
# print('Training done.')

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='train')
plt.plot(val_losses,   label='val')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Image→Music Training Loss')
plt.legend(); plt.grid(True)
plt.savefig(f'{EVAL_DIR}/loss_curve.png', dpi=150)
plt.show()

## Save Checkpoint

In [ ]:
SAVE_DIR = '/content/drive/MyDrive/18789 Project/image_to_music_checkpoint'
os.makedirs(SAVE_DIR, exist_ok=True)
musicgen.save_pretrained(SAVE_DIR)
musicgen_processor.save_pretrained(SAVE_DIR)
torch.save({
    'projection':       projection.state_dict(),
    'inst_clf':         inst_clf.state_dict(),
    'clip_dim':         CLIP_DIM,
    'text_enc_dim':     TEXT_ENC_DIM,
    'seq_len':          SEQ_LEN,
    'instrument_names': INSTRUMENT_NAMES,
    'train_losses':     train_losses,
    'val_losses':       val_losses,
}, os.path.join(SAVE_DIR, 'heads.pt'))
print('Saved to', SAVE_DIR)

## Load Checkpoint
Run this cell **only** if Colab disconnected mid-session. Skip if model is still in memory.

In [ ]:
SAVE_DIR = '/content/drive/MyDrive/18789 Project/image_to_music_checkpoint'

musicgen           = MusicgenForConditionalGeneration.from_pretrained(SAVE_DIR).to(device)
musicgen_processor = AutoProcessor.from_pretrained(SAVE_DIR)
for cfg in [musicgen.config, musicgen.decoder.config, musicgen.generation_config]:
    cfg.decoder_start_token_id = 2048
    cfg.pad_token_id           = 2048

heads      = torch.load(os.path.join(SAVE_DIR, 'heads.pt'), map_location=device)
projection = ImageProjection(heads['clip_dim'], heads['text_enc_dim'], heads['seq_len']).to(device)
projection.load_state_dict(heads['projection'])
inst_clf   = InstrumentClassifier(heads['clip_dim'], len(heads['instrument_names'])).to(device)
inst_clf.load_state_dict(heads['inst_clf'])
train_losses = heads.get('train_losses', [])
val_losses   = heads.get('val_losses', [])
print('Checkpoint loaded.')

## Inference

In [ ]:
def generate_from_image(image_path, duration_seconds=15):
    """Main inference function. Returns (audio_np, sr, instrument_predictions)."""
    musicgen.eval(); projection.eval(); inst_clf.eval()
    with torch.no_grad():
        clip_feats, enc_out, attn_mask = image_to_encoder_outputs(image_path)
        audio = musicgen.generate(
            encoder_outputs=enc_out, attention_mask=attn_mask,
            do_sample=True, guidance_scale=1.0,
            max_new_tokens=int(duration_seconds * musicgen.config.audio_encoder.frame_rate),
        )
        probs = torch.sigmoid(inst_clf(clip_feats))[0]
    sr   = musicgen.config.audio_encoder.sampling_rate
    preds = [(INSTRUMENT_NAMES[i], probs[i].item()) for i in probs.argsort(descending=True)]
    return audio[0, 0].cpu().numpy(), sr, preds

def generate_from_text(prompt, duration_seconds=15):
    """Text baseline. CFG=3.0 works fine for the standard text path."""
    musicgen.eval()
    inputs = musicgen_processor(text=[prompt], padding=True, return_tensors='pt').to(device)
    with torch.no_grad():
        audio = musicgen.generate(
            **inputs, do_sample=True, guidance_scale=3.0,
            max_new_tokens=int(duration_seconds * musicgen.config.audio_encoder.frame_rate),
        )
    return audio[0, 0].cpu().numpy(), musicgen.config.audio_encoder.sampling_rate

In [ ]:
# Quick test — change path to any image in your dataset
TEST_IMAGE = list(eval_samples.values())[0]['image_path']

audio_np, sr, preds = generate_from_image(TEST_IMAGE, duration_seconds=10)
print('Predicted instruments:')
for name, prob in preds[:6]:
    print(f'  {name:<12} {"█" * int(prob*20)} {prob:.2f}')
save_wav(audio_np, sr, f'{EVAL_DIR}/quick_test.wav')
print(f'Saved quick_test.wav  ({len(audio_np)/sr:.1f}s)')

## Post-Training Evaluation

In [ ]:
from transformers import ClapModel, AutoProcessor as ClapAutoProcessor
from sklearn.neighbors import KNeighborsClassifier

clap_model     = ClapModel.from_pretrained('laion/clap-htsat-unfused').to(device)
clap_processor = ClapAutoProcessor.from_pretrained('laion/clap-htsat-unfused')

def clap_embed_file(path, target_sr=48000, duration=10):
    waveform, sr = torchaudio.load(path)
    if waveform.shape[0] > 1: waveform = waveform.mean(dim=0, keepdim=True)
    if sr != target_sr: waveform = torchaudio.transforms.Resample(sr, target_sr)(waveform)
    n = target_sr * duration
    waveform = waveform[:, :n] if waveform.shape[1] >= n else nn.functional.pad(waveform, (0, n - waveform.shape[1]))
    inputs = clap_processor(audio=waveform.squeeze(0).numpy(), sampling_rate=target_sr, return_tensors='pt').to(device)
    with torch.no_grad(): emb = clap_model.get_audio_features(**inputs)[0].cpu().numpy()
    return emb / (np.linalg.norm(emb) + 1e-8)

def clap_embed_numpy(audio_np, sr, target_sr=48000, duration=10):
    if sr != target_sr:
        audio_np = scipy.signal.resample(audio_np, int(len(audio_np) * target_sr / sr))
    n = target_sr * duration
    audio_np = audio_np[:n] if len(audio_np) >= n else np.pad(audio_np, (0, n - len(audio_np)))
    inputs = clap_processor(audio=audio_np.astype(np.float32), sampling_rate=target_sr, return_tensors='pt').to(device)
    with torch.no_grad(): emb = clap_model.get_audio_features(**inputs)[0].cpu().numpy()
    return emb / (np.linalg.norm(emb) + 1e-8)

print('CLAP loaded.')

In [ ]:
# ── Metric 1: CLAP similarity vs reference audio ──────────────────────────────
print('Generating post-training outputs + computing CLAP similarity...\n')
results = []
for culture, sample in eval_samples.items():
    ref_emb = clap_embed_file(sample['audio_path'])
    ba_emb  = clap_embed_file(f'{EVAL_DIR}/{culture}_baseline_a_pretrain.wav')
    bb_emb  = clap_embed_file(f'{EVAL_DIR}/{culture}_baseline_b_text.wav')

    our_np, our_sr = generate_from_image_raw(sample['image_path'])
    save_wav(our_np, our_sr, f'{EVAL_DIR}/{culture}_ours_finetuned.wav')
    our_emb = clap_embed_numpy(our_np, our_sr)

    results.append({
        'culture':    culture,
        'baseline_a': float(np.dot(ref_emb, ba_emb)),
        'baseline_b': float(np.dot(ref_emb, bb_emb)),
        'ours':       float(np.dot(ref_emb, our_emb)),
    })

print(f'{"Culture":<14} {"Baseline A":>12} {"Baseline B":>12} {"Ours":>12}')
print('-' * 52)
for r in results:
    print(f'{r["culture"]:<14} {r["baseline_a"]:>12.4f} {r["baseline_b"]:>12.4f} {r["ours"]:>12.4f}')
avg_a, avg_b, avg_our = [np.mean([r[k] for r in results]) for k in ['baseline_a','baseline_b','ours']]
print('-' * 52)
print(f'{"Average":<14} {avg_a:>12.4f} {avg_b:>12.4f} {avg_our:>12.4f}')
print(f'\nImprovement over Baseline A: {(avg_our-avg_a)*100:+.2f}pp')
print(f'Improvement over Baseline B: {(avg_our-avg_b)*100:+.2f}pp')

In [ ]:
# ── Metric 2: Culture classifier on generated audio ────────────────────────────
culture_labels = list(eval_samples.keys())
culture_to_idx = {c: i for i, c in enumerate(culture_labels)}

ref_embs, ref_lbls = [], []
for i in val_ds.indices:
    row = dataset.data[i]
    ref_embs.append(clap_embed_file(row['music_path']))
    ref_lbls.append(culture_to_idx[row['culture'].strip().lower()])

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(ref_embs, ref_lbls)

def classify_file(path):
    return culture_labels[knn.predict(clap_embed_file(path).reshape(1,-1))[0]]

print(f'{"Culture":<14} {"Expected":<14} {"Baseline A":<16} {"Baseline B":<16} {"Ours":<16}')
print('-' * 76)
ca = cb = co = 0
for culture in eval_samples:
    pa = classify_file(f'{EVAL_DIR}/{culture}_baseline_a_pretrain.wav')
    pb = classify_file(f'{EVAL_DIR}/{culture}_baseline_b_text.wav')
    po = classify_file(f'{EVAL_DIR}/{culture}_ours_finetuned.wav')
    t  = lambda p: '✓' if p == culture else '✗'
    print(f'{culture:<14} {culture:<14} {pa+" "+t(pa):<16} {pb+" "+t(pb):<16} {po+" "+t(po):<16}')
    ca += (pa==culture); cb += (pb==culture); co += (po==culture)
n = len(eval_samples)
print('-' * 76)
print(f'{"Accuracy":<28} {ca}/{n}             {cb}/{n}             {co}/{n}')

In [ ]:
# ── Metric 3: Instrument classifier accuracy on val images ────────────────────
projection.eval(); inst_clf.eval()
correct = total = 0
per_culture = {c: [0,0] for c in CULTURE_INSTRUMENTS}

for i in val_ds.indices:
    row     = dataset.data[i]
    culture = row['culture'].strip().lower()
    image   = Image.open(row['image_path']).convert('RGB')
    with torch.no_grad():
        probs = torch.sigmoid(inst_clf(get_clip_features([image])))[0]
    pred = max(CULTURE_INSTRUMENTS, key=lambda c: probs[CULTURE_INSTRUMENTS[c]].mean().item())
    correct += (pred == culture); total += 1
    per_culture[culture][0] += (pred == culture); per_culture[culture][1] += 1

print(f'Instrument classifier accuracy: {correct}/{total} = {correct/total:.1%}')
for c, (hit, tot) in per_culture.items():
    if tot: print(f'  {c:<14}: {hit}/{tot} = {hit/tot:.1%}')

In [ ]:
# ── Summary bar chart ─────────────────────────────────────────────────────────
x = np.arange(len(results)); w = 0.25
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x-w, [r['baseline_a'] for r in results], w, label='Baseline A (untrained image)', color='#d9534f')
ax.bar(x,   [r['baseline_b'] for r in results], w, label='Baseline B (text prompt)',     color='#f0ad4e')
ax.bar(x+w, [r['ours']       for r in results], w, label='Ours (fine-tuned image)',      color='#5cb85c')
ax.set_ylabel('CLAP Cosine Similarity vs Reference')
ax.set_title('Cultural Music Generation: Before vs After Fine-Tuning')
ax.set_xticks(x)
ax.set_xticklabels([r['culture'].replace('_',' ').title() for r in results])
ax.legend(); ax.set_ylim(0,1); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{EVAL_DIR}/evaluation_summary.png', dpi=150)
plt.show()

## Gradio Demo

In [ ]:
import gradio as gr, tempfile

DEMO_PROMPTS = {
    'Auto-detect':  None,
    'Indian':       'Indian classical music with sitar and tabla',
    'Middle East':  'traditional Arabic music with oud and qanun',
    'East Asian':   'traditional Chinese music with guzheng and erhu',
}
INST_TO_PROMPT = {
    **{i: DEMO_PROMPTS['Indian']      for i in ['sitar','tabla','sarod','bansuri','mridangam','dhol','dholak','tasha']},
    **{i: DEMO_PROMPTS['Middle East'] for i in ['oud','qanun','ney','darbuka','riq','rabab']},
    **{i: DEMO_PROMPTS['East Asian']  for i in ['guzheng','erhu','pipa','dizi','taiko','shamisen','koto']},
}

def run_demo(image_pil, duration, culture_override):
    if image_pil is None: return None, None, 'Please upload an image.'
    with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as f:
        image_pil.save(f.name); tmp = f.name

    audio_np, sr, preds = generate_from_image(tmp, duration_seconds=int(duration))
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
        save_wav(audio_np, sr, f.name); img_path = f.name

    prompt = DEMO_PROMPTS.get(culture_override) or INST_TO_PROMPT.get(preds[0][0], DEMO_PROMPTS['Indian'])
    txt_np, txt_sr = generate_from_text(prompt, duration_seconds=int(duration))
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
        save_wav(txt_np, txt_sr, f.name); txt_path = f.name

    inst_md = '**Predicted Instruments:**\n' + '\n'.join(f'- {n}: {p:.0%}' for n,p in preds[:8])
    inst_md += f'\n\n*Text baseline:* `{prompt}`'
    return img_path, txt_path, inst_md

with gr.Blocks(title='Image → Music') as demo:
    gr.Markdown('# Image → Music\nUpload a cultural image to generate matching traditional music.')
    with gr.Row():
        with gr.Column():
            img_in  = gr.Image(type='pil', label='Upload Image')
            dur_sl  = gr.Slider(5, 30, value=15, step=5, label='Duration (s)')
            cult_dd = gr.Dropdown(list(DEMO_PROMPTS.keys()), value='Auto-detect', label='Culture override')
            btn     = gr.Button('Generate', variant='primary')
        with gr.Column():
            inst_out = gr.Markdown()
            aud_img  = gr.Audio(label='Our Model (Image→Music)', type='filepath')
            aud_txt  = gr.Audio(label='Text Baseline',           type='filepath')
    btn.click(run_demo, [img_in, dur_sl, cult_dd], [aud_img, aud_txt, inst_out])

demo.launch(share=True, debug=True)